<a href="https://colab.research.google.com/github/shashankk2807/rpc.py/blob/main/internal_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langgraph langchain-groq langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 981.3 kB/s eta 0:00:00


In [2]:
# =====================================================================
# Step 2: Import Necessary Libraries
# =====================================================================
import os
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from google.colab import userdata

In [3]:
# =====================================================================
# Step 3: Initialization & Setup (Using an Active Groq Model)
# =====================================================================
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Updated to an active model on Groq's free tier
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2)

# Set up the internal memory checkpointer needed for interrupts
checkpointer = InMemorySaver()

In [4]:
# Step 4: Define the Graph Nodes
# =====================================================================
def llm_node(state: MessagesState):
    """Generates or revises content based on the message history."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def human_node(state: MessagesState):
    """Displays the AI output and triggers an interrupt for feedback."""
    last_ai_message = state["messages"][-1].content

    # Note: Groq returns a string directly, whereas Google GenAI sometimes returned a list structure.
    print(f"\n--- AI Generated Content ---\n{last_ai_message}\n----------------------------")

    # The interrupt pauses graph execution here and surfaces the prompt.
    feedback = interrupt(
        "Action (Type 'APPROVE' to finish, or type your revision feedback): "
    )

    # Save the human feedback as a message in the state
    return {"messages": [HumanMessage(content=feedback)]}

In [5]:
# Step 5: Define the Conditional Routing Logic
# =====================================================================
def router_logic(state: MessagesState):
    """Inspects human feedback to decide whether to exit or loop back."""
    last_user_message = state["messages"][-1].content

    if last_user_message.strip().upper() == "APPROVE":
        return "accept"

    # If it's anything else, route back to the LLM node for revision
    return "revise"

In [6]:
# Step 6: Assemble the Graph Layout with Checkpointer Configuration
# =====================================================================
builder = StateGraph(MessagesState)

builder.add_node("llm_worker", llm_node)
builder.add_node("human_gate", human_node)

builder.add_edge(START, "llm_worker")
builder.add_edge("llm_worker", "human_gate")

# Add conditional edge to handle routing decisions
builder.add_conditional_edges(
    "human_gate",
    router_logic,
    {
        "accept": END,
        "revise": "llm_worker"
    }
)

# Crucial: Compile the graph with a checkpointer to save state at interrupts
compiled_graph = builder.compile(checkpointer=checkpointer)

In [7]:
# Step 7: Runtime Execution via Single-Loop Thread Management
# =====================================================================
print("--- Starting Human-in-the-Loop Loopback Graph (Groq Powered) ---")
print("Instructions: Enter a Thread ID to start or resume a session. Type 'exit' to quit.")

while True:
    # 1. Select your active thread context
    thread_id = input("\nEnter Thread ID (e.g., user_1, user_2) or 'exit': ").strip()

    if thread_id.lower() == 'exit':
        print("Shutting down conversational engine.")
        break

    if not thread_id:
        continue

    config = {"configurable": {"thread_id": thread_id}}
    current_state = compiled_graph.get_state(config)

    # 2. If the thread has an active next node and contains a pending interrupt
    if current_state.next and current_state.tasks and getattr(current_state.tasks[0], "interrupts", None):
        print(f" >> Resuming active thread: '{thread_id}'")
        prompt_string = current_state.tasks[0].interrupts[0].value
        user_response = input(prompt_string).strip()

        # Advance graph using Command
        compiled_graph.invoke(Command(resume=user_response), config=config)

    # 3. If the thread is brand new or completely finished (no next node steps remaining)
    else:
        print(f" >> Initializing new session for thread: '{thread_id}'")
        user_idea = input("Enter your idea: ").strip()

        if not user_idea:
            continue

        initial_input = {"messages": [HumanMessage(content=user_idea)]}
        compiled_graph.invoke(initial_input, config=config)

    # 4. Fetch a fresh snapshot to check metadata safely using get_state
    post_run_state = compiled_graph.get_state(config)
    if not post_run_state.next:
        print(f" >> Workflow for thread '{thread_id}' completed successfully via APPROVE!")

--- Starting Human-in-the-Loop Loopback Graph (Groq Powered) ---
Instructions: Enter a Thread ID to start or resume a session. Type 'exit' to quit.

Enter Thread ID (e.g., user_1, user_2) or 'exit': test_1
 >> Initializing new session for thread: 'test_1'
Enter your idea: Write a one-liner greeting.

--- AI Generated Content ---
Hello! Hope your day shines as brightly as your smile!
----------------------------

Enter Thread ID (e.g., user_1, user_2) or 'exit': test_1
 >> Resuming active thread: 'test_1'
Action (Type 'APPROVE' to finish, or type your revision feedback): Approve

--- AI Generated Content ---
Hello! Hope your day shines as brightly as your smile!
----------------------------
 >> Workflow for thread 'test_1' completed successfully via APPROVE!

Enter Thread ID (e.g., user_1, user_2) or 'exit': test_2
 >> Initializing new session for thread: 'test_2'
Enter your idea: Write a short python function to add two numbers.

--- AI Generated Content ---
Here’s a tiny, self‑contain